# 03 — VAE Anomaly Detection

This notebook trains a Variational Autoencoder (VAE) on normal IoT traffic and evaluates anomaly detection on the test split using reconstruction error.

In [ ]:
%pip install torch

# Cell 1: Imports and path setup
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, roc_curve

import torch
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from backend.models.vae import VAE, DEFAULT_CONFIG, DEVICE, vae_loss, anomaly_score

print(f'Using device: {DEVICE}')

In [ ]:
# Cell 2: Load processed arrays
processed_dir_candidates = [
    project_root / 'backend' / 'data' / 'processed',
    project_root / 'data' / 'processed',
]

processed_dir = next((p for p in processed_dir_candidates if p.exists()), None)
if processed_dir is None:
    raise FileNotFoundError('No processed data directory found.')

X_train = np.load(processed_dir / 'X_train.npy')
X_val = np.load(processed_dir / 'X_val.npy')
X_test = np.load(processed_dir / 'X_test.npy')
y_train = np.load(processed_dir / 'y_train.npy') if (processed_dir / 'y_train.npy').exists() else None
y_val = np.load(processed_dir / 'y_val.npy') if (processed_dir / 'y_val.npy').exists() else None
y_test = np.load(processed_dir / 'y_test.npy')

print('Processed dir:', processed_dir)
print('X_train:', X_train.shape)
print('X_val:', X_val.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

In [ ]:
# Cell 3: Keep normal traffic for training and validation
if y_train is not None and len(y_train) == len(X_train):
    X_train_normal = X_train[y_train == 0]
else:
    X_train_normal = X_train

if y_val is not None and len(y_val) == len(X_val):
    X_val_normal = X_val[y_val == 0]
else:
    X_val_normal = X_val

print('Normal train windows:', X_train_normal.shape)
print('Normal val windows:', X_val_normal.shape)

In [ ]:
# Cell 4: Train VAE
batch_size = 64
epochs = 20
learning_rate = 1e-3
beta = 1.0

train_loader = DataLoader(TensorDataset(torch.tensor(X_train_normal, dtype=torch.float32)), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val_normal, dtype=torch.float32)), batch_size=batch_size, shuffle=False)

config = DEFAULT_CONFIG.copy()
config.update({
    'seq_len': int(X_train.shape[1]),
    'n_features': int(X_train.shape[2]),
    'latent_dim': 64,
    'beta': beta,
})

model = VAE(config=config).to(DEVICE)
optimizer = Adam(model.parameters(), lr=learning_rate)

history = {'train_loss': [], 'val_loss': [], 'train_recon': [], 'val_recon': [], 'train_kl': [], 'val_kl': []}
best_val = float('inf')
weights_path = project_root / 'backend' / 'models' / 'weights' / 'vae.pt'

for epoch in range(1, epochs + 1):
    model.train()
    train_loss_sum = 0.0
    train_recon_sum = 0.0
    train_kl_sum = 0.0
    train_count = 0

    for (batch_x,) in train_loader:
        batch_x = batch_x.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        x_recon, mu, log_var = model(batch_x)
        loss, recon_loss, kl_loss = vae_loss(x_recon, batch_x, mu, log_var, beta=beta)
        loss.backward()
        optimizer.step()

        bs = batch_x.size(0)
        train_count += bs
        train_loss_sum += loss.item() * bs
        train_recon_sum += recon_loss.item() * bs
        train_kl_sum += kl_loss.item() * bs

    model.eval()
    val_loss_sum = 0.0
    val_recon_sum = 0.0
    val_kl_sum = 0.0
    val_count = 0

    with torch.no_grad():
        for (batch_x,) in val_loader:
            batch_x = batch_x.to(DEVICE)
            x_recon, mu, log_var = model(batch_x)
            loss, recon_loss, kl_loss = vae_loss(x_recon, batch_x, mu, log_var, beta=beta)

            bs = batch_x.size(0)
            val_count += bs
            val_loss_sum += loss.item() * bs
            val_recon_sum += recon_loss.item() * bs
            val_kl_sum += kl_loss.item() * bs

    train_loss = train_loss_sum / max(train_count, 1)
    train_recon = train_recon_sum / max(train_count, 1)
    train_kl = train_kl_sum / max(train_count, 1)
    val_loss = val_loss_sum / max(val_count, 1)
    val_recon = val_recon_sum / max(val_count, 1)
    val_kl = val_kl_sum / max(val_count, 1)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_recon'].append(train_recon)
    history['val_recon'].append(val_recon)
    history['train_kl'].append(train_kl)
    history['val_kl'].append(val_kl)

    if val_loss < best_val:
        best_val = val_loss
        model.save_weights(weights_path)

    print(f'Epoch {epoch:02d}/{epochs} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}')

print('Saved best weights to:', weights_path)

In [ ]:
# Cell 5: Plot training curves
plt.figure(figsize=(12, 5))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('VAE Total Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Compute anomaly scores and threshold
best_model = VAE.from_weights(weights_path)
train_scores = anomaly_score(best_model, X_train_normal, DEVICE)
test_scores = anomaly_score(best_model, X_test, DEVICE)

threshold = float(np.percentile(train_scores, 95))
y_pred = (test_scores >= threshold).astype(int)

print(f'Threshold (95th percentile of normal train scores): {threshold:.6f}')

In [ ]:
# Cell 7: Evaluation metrics and ROC curve
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
roc_auc = roc_auc_score(y_test, test_scores)

print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1:        {f1:.4f}')
print(f'ROC AUC:   {roc_auc:.4f}')

fpr, tpr, _ = roc_curve(y_test, test_scores)
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'VAE (AUC={roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('ROC Curve — VAE Anomaly Detector')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Reconstruction error distributions
normal_scores = test_scores[y_test == 0]
anomaly_scores = test_scores[y_test == 1]

plt.figure(figsize=(10, 5))
plt.hist(normal_scores, bins=50, alpha=0.6, label='Normal', density=True)
plt.hist(anomaly_scores, bins=50, alpha=0.6, label='Anomaly', density=True)
plt.axvline(threshold, color='red', linestyle='--', label='Threshold')
plt.title('Reconstruction Error Distribution (Test Set)')
plt.xlabel('MSE Reconstruction Error')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()